# 04 - FPMC (Factorized Personalized Markov Chains)

FPMC scores *"given this traveller, currently in city `l`, how likely is city `i` next?"* by adding two factorized terms:

$$\text{score}(u,l,i) = \underbrace{\langle V^{UI}_u, V^{IU}_i\rangle}_{\text{personalization}} + \underbrace{\langle V^{LI}_l, V^{IL}_i\rangle}_{\text{Markov transition}}$$

The first term is matrix factorization (who prefers what); the second is a factorized Markov chain (what follows what). For the benchmark it is trained with **full-catalogue cross-entropy** and **AdamW**, the same objective and optimiser as GRU4Rec and SASRec, so differences between them come from the architecture rather than the training setup (the classic BPR loss is still available via `loss="bpr"`).

### Benchmark protocol (same as GRU4Rec / Markov)
1. **Tune** with seed 42: train on Train, select on validation **NDCG@10** -- 25 Optuna TPE trials (median pruner), full-catalogue cross-entropy, AdamW.
2. **Record $E_{best}$**, the peak epoch of the winning trial.
3. **Merge** Train + Validation.
4. **Train 5 fresh models** (seeds `[1, 7, 13, 2024, 99]`) for exactly $E_{best}$ epochs, no early stopping.
5. **Score on Test**; log **mean** and **std** across seeds to `results.csv`.

**Search space.** FPMC gets the identical 25-trial budget, but only the knobs of the unified space that exist in it are searched: `embed_dim` {64,128,256}, `lr` [1e-4,1e-2] log, `weight_decay` [1e-6,1e-3] log, `batch_size` {64,128,256}. FPMC has no layers, no dropout and only reads the last city, so `num_layers`, `dropout` and `max_len` do not apply.

This is done twice -- **persona** and **user** personalization axis -- each with its own search and the identical budget.

All logic is in `src/models/fpmc.py`. Running this notebook top to bottom does everything and ends by displaying `results.csv`.

> **Runtime:** 2 variants x 25 trials, then 2 x 5 final fits. Use a GPU. To smoke-test the pipeline first, set `N_TRIALS = 2` and `SEEDS = SEEDS[:2]` in the config cell below -- and don't trust those numbers.

In [1]:
import torch

import os, sys, json
def find_root(start=None):
    d = os.path.abspath(start or os.getcwd())
    for _ in range(6):
        if os.path.exists(os.path.join(d, 'src', 'preprocessing.py')):
            return d
        d = os.path.dirname(d)
    return os.path.abspath(os.path.join(os.getcwd(), '..'))
ROOT = find_root(); sys.path.insert(0, ROOT)
import pandas as pd
from src.preprocessing import load_processed, preprocess
from src.evaluate import append_result
try:
    data = load_processed(os.path.join(ROOT, 'data', 'processed'))
except FileNotFoundError:
    data = preprocess(data_dir=os.path.join(ROOT, 'data'), sample_n_trips=5000)
print('loaded:', data['meta'])
if data['meta'].get('sample_n_trips') is not None:
    print('WARNING: processed.pkl was built on a subsample (sample_n_trips=%r). '
          'Delete data/processed/processed.pkl and rebuild with sample_n_trips=None '
          'before trusting or logging any numbers.' % data['meta']['sample_n_trips'])

RESULTS = os.path.join(ROOT, 'results.csv')
TUNING_DIR = os.path.join(ROOT, 'results', 'tuning'); os.makedirs(TUNING_DIR, exist_ok=True)

if torch.cuda.is_available():
    DEVICE = 'cuda'
elif torch.backends.mps.is_available():          # Apple Silicon GPU
    DEVICE = 'mps'
else:
    DEVICE = 'cpu'
print('device:', DEVICE)

loaded: {'min_trip_len': 4, 'val_frac': 0.1, 'sample_n_trips': None, 'seed': 42, 'n_train_trips': 190737, 'n_valid_trips': 21187, 'n_test_trips': 70662, 'n_cities': 39903, 'n_users': 200153, 'n_personas': 15, 'n_test_targets_unseen': 0}
device: mps


In [2]:
from src.models.fpmc import tune_fpmc as TUNE, final_fpmc as FINAL
from src.models.gru4rec import N_TRIALS, FINAL_SEEDS, MAX_EPOCHS, PATIENCE
SEEDS = list(FINAL_SEEDS)
# N_TRIALS = 2; SEEDS = SEEDS[:2]        # <- uncomment for a quick smoke test only
print(f"trials/variant={N_TRIALS} | max_epochs={MAX_EPOCHS} | patience={PATIENCE} | seeds={SEEDS}")

trials/variant=25 | max_epochs=20 | patience=3 | seeds=[1, 7, 13, 2024, 99]


## Ablation - personalization axis: raw `user_id` vs `persona`

Classic FPMC personalizes by user. But in this dataset almost every `user_id` occurs in a single trip, so a per-user factor cannot generalise to unseen test users. We therefore also try a **persona** = (booker_country, device_class), a coarse profile that recurs across many trips. Comparing the two is a headline finding: it shows *where personalization signal actually lives* in this data.

For each variant: **tune** (validation only) -> read $E_{best}$ -> **final fits** on Train + Val for 5 seeds -> append the mean row and the std row to `results.csv`, and save the per-seed table and chosen hyper-parameters under `results/tuning/`.

Rows written: `FPMC-persona`, `FPMC-persona (std)`, `FPMC-user`, `FPMC-user (std)`.

In [3]:
def run_variant(tag, row_name, tune_kw):
    """Steps 1-5 for one variant: tune on validation, read E_best, 5-seed final
    fits on train+val, save params + per-seed table, log mean and std rows."""
    print(f"\n==================== {row_name} ====================")

    # steps 1-2: search on validation, record E_best
    study = TUNE(data, n_trials=N_TRIALS, device=DEVICE, show_progress_bar=False, **tune_kw)
    best_params = dict(study.best_params)
    e_best = study.best_trial.user_attrs["best_epoch"]
    with open(os.path.join(TUNING_DIR, f"{tag}.json"), "w") as f:
        json.dump({"best_params": best_params, "E_best": e_best,
                   "best_val_NDCG@10": study.best_value, "n_trials": len(study.trials)}, f, indent=2)

    # steps 3-5: train+val, 5 seeds x E_best epochs, score on test
    per_seed, _ = FINAL(data, best_params, epochs=e_best, seeds=SEEDS, device=DEVICE, **tune_kw)
    per_seed.to_csv(os.path.join(TUNING_DIR, f"{tag}-per-seed.csv"))

    mean, std = per_seed.mean(), per_seed.std()
    mean_row = {k: (int(v) if k == "n_params" else float(v)) for k, v in mean.items()}
    std_row = {k: float(v) for k, v in std.items()}
    append_result(RESULTS, row_name, mean_row)
    append_result(RESULTS, f"{row_name} (std)", std_row)
    summaries[row_name] = {"best_params": best_params, "E_best": e_best,
                           "val_NDCG@10": round(study.best_value, 4),
                           "test_NDCG@10": f"{mean['NDCG@10']:.4f} +/- {std['NDCG@10']:.4f}"}

summaries = {}

### Persona

In [4]:
run_variant("fpmc-persona", "FPMC-persona", {"axis": "persona"})


==================== FPMC-persona ====================


/Users/ameensane/Desktop/SC4020/SC4020-PROJECT1/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


FPMC (persona): best val NDCG@10 = 0.4377 (7/25 completed, 18 pruned, 17566s)
  params: {'lr': 0.0004059611610484307, 'weight_decay': 3.752055855124284e-05, 'embed_dim': 256, 'batch_size': 256}
  best epoch (E_best): 8



FPMC (persona) - 5 seeds
                    mean     std
HR@1        2.448000e-01  0.0005
NDCG@1      2.448000e-01  0.0005
Accuracy@4  4.438000e-01  0.0007
HR@5        4.764000e-01  0.0006
NDCG@5      3.669000e-01  0.0004
HR@10       5.763000e-01  0.0010
NDCG@10     3.992000e-01  0.0005
MRR         3.439000e-01  0.0004
n_trips     7.066200e+04  0.0000
train_secs  1.579200e+03  0.9670
n_params    3.065114e+07  0.0000


### User

In [5]:
run_variant("fpmc-user", "FPMC-user", {"axis": "user"})


==================== FPMC-user ====================


FPMC (user): best val NDCG@10 = 0.4172 (14/25 completed, 11 pruned, 42821s)
  params: {'lr': 0.0007465149640099093, 'weight_decay': 1.3926171590609833e-06, 'embed_dim': 256, 'batch_size': 256}
  best epoch (E_best): 2


**Note on FPMC-user.** The 25-trial search above finished (best params and E_best = 2 saved to `results/tuning/fpmc-user.json`), but the run was stopped during the 5-seed final fits. Steps 3-5 were then run separately with the same `final_fpmc` call and the saved params (seeds 1, 7, 13, 2024, 99, 2 epochs on train+val, scored on test). Per-seed results: `results/tuning/fpmc-user-per-seed.csv`; mean and std rows: `results.csv`.

Test NDCG@10 = **0.3897 +/- 0.0005**.

## Selected configurations

In [ ]:
display(pd.DataFrame(summaries).T)

## Final results (`results.csv`)

In [ ]:
display(pd.read_csv(RESULTS))

## The position feature

FPMC has no built-in notion of *where* in a trip a step sits (SASRec learns this for free via positional embeddings). We add a position-bucket embedding to the user side of the score, so the model can express e.g. "early in a trip vs the last leg home". Its effect is included in the numbers above.

## What to write up

- FPMC should comfortably beat Popularity and Markov wherever the last-city transition and personalization carry signal.
- Report the **persona vs user** gap as mean +/- std over 5 seeds (a gap smaller than the std is not a real effect), and explain it with the per-user sparsity measured in `01_data_exploration.ipynb`.
- Expect the raw-`user_id` variant to degrade (cold-start at test), which is the point of the ablation.
- Note in the report that FPMC was searched over the 4 knobs of the unified space that exist in it, with the same 25-trial budget as every other model.